# Deploy RAG Agent (dev)
Este notebook **no genera** `rag_agent.py`. Asume que ya existe en:

`/Workspace/Users/<tu_usuario>/bind_agent/rag/rag_agent.py`

Objetivo:
1) Log + register del modelo en **Unity Catalog** (MLflow registry).
2) Crear/actualizar un **Model Serving endpoint** que sirva ese modelo.
3) Probar una invocación simple.

> Nota: en Compute Serverless suele faltar `mlflow`; por eso instalamos dependencias en la primera celda.


In [0]:
# # ==============================
# # 1) Dependencias (solo para pruebas locales)
# # ==============================
# En Serverless compute puede no venir mlflow instalado.
%pip install -U databricks-vectorsearch mlflow databricks-sdk
dbutils.library.restartPython()


In [0]:
# ==============================
# 2) Config (solo para pruebas locales, en prod se toman del asset bundle)
# ==============================
from pathlib import Path
import time

# --- Vector Search ---
VS_ENDPOINT = "bind_agent_vs"
VS_INDEX_FULL_NAME = "bind_agent.docs.pdf_chunks_vs_idx"

# --- Endpoints ---
EMBED_ENDPOINT = "databricks-bge-large-en"  # for query_vector
EMB_ENDPOINT = EMBED_ENDPOINT  # backward-compatible alias
LLM_ENDPOINT = "databricks-gemma-3-12b"

# --- Retrieval params ---
TOP_K_CANDIDATES = "40"
TOP_K_FINAL = "8"
LEX_FALLBACK_LIMIT = "20"

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS = "14000"
RERANK_SNIPPET_CHARS = "1200"

# --- LLM params ---
TEMPERATURE_RERANK = "0.0"
TEMPERATURE_ANSWER = "0.2"
MAX_TOKENS_ANSWER = "900"

# --- Retry params ---
MAX_RETRIES = "3"
RETRY_SLEEP_SECS = "1.0"

# MLflow / UC
# Recomendación: catalog.schema.model
UC_MODEL_NAME = "bind_agent.docs.rag_agent"

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/rag_agent_deploy"

# Serving endpoint para el agente
MODEL_SERVING_ENDPOINT = "bind_agent_rag_agent"


In [0]:
import os, importlib.util
from pathlib import Path

In [0]:
def p(name, default):
    try:
        v = dbutils.widgets.get(name)
        return v if v not in (None, "") else default
    except:
        return default

BUNDLE_FILE_PATH = p("BUNDLE_FILE_PATH", "")
print(BUNDLE_FILE_PATH)

# --- Vector Search ---
VS_ENDPOINT = p("VS_ENDPOINT", "bind_agent_vs")
VS_INDEX_FULL_NAME = p("VS_INDEX_FULL_NAME", "bind_agent.docs.pdf_chunks_vs_idx")

# --- Endpoints ---
EMBED_ENDPOINT = p("EMBED_ENDPOINT", "databricks-bge-large-en")
LLM_ENDPOINT   = p("LLM_ENDPOINT", "databricks-gemma-3-12b")

# --- Retrieval params ---
TOP_K_CANDIDATES    = p("TOP_K_CANDIDATES", "40")
TOP_K_FINAL         = p("TOP_K_FINAL", "8")
LEX_FALLBACK_LIMIT  = p("LEX_FALLBACK_LIMIT", "20")

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS     = p("MAX_CONTEXT_CHARS", "14000")
RERANK_SNIPPET_CHARS  = p("RERANK_SNIPPET_CHARS", "1200")

# --- LLM params ---
TEMPERATURE_RERANK  = p("TEMPERATURE_RERANK", "0.0")
TEMPERATURE_ANSWER  = p("TEMPERATURE_ANSWER", "0.2")
MAX_TOKENS_ANSWER   = p("MAX_TOKENS_ANSWER", "900")

# --- Retry params ---
MAX_RETRIES        = p("MAX_RETRIES", "3")
RETRY_SLEEP_SECS   = p("RETRY_SLEEP_SECS", "1.0")

# UC model
UC_MODEL_NAME = p("UC_MODEL_NAME", "bind_agent.docs.rag_agent")

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/rag_agent_deploy"

# Serving endpoint
MODEL_SERVING_ENDPOINT = p("MODEL_SERVING_ENDPOINT", "bind_agent_rag_agent")

In [0]:
# # ==============================
# # 2.1) Config para rag y serving endpoint
# # ==============================

# 0) Setear env vars usadas en el RAG
os.environ["RAG_LLM_ENDPOINT"] = LLM_ENDPOINT
os.environ["RAG_EMBED_ENDPOINT"] = EMBED_ENDPOINT
os.environ["RAG_VS_ENDPOINT"] = VS_ENDPOINT
os.environ["RAG_VS_INDEX"] = VS_INDEX_FULL_NAME
os.environ["RAG_VS_INDEX_FULL_NAME"] = VS_INDEX_FULL_NAME
os.environ["RAG_TOP_K_CANDIDATES"] = TOP_K_CANDIDATES
os.environ["RAG_TOP_K_FINAL"] = TOP_K_FINAL
os.environ["RAG_LEX_FALLBACK_LIMIT"] = LEX_FALLBACK_LIMIT
os.environ["RAG_MAX_CONTEXT_CHARS"] = MAX_CONTEXT_CHARS
os.environ["RAG_RERANK_SNIPPET_CHARS"] = RERANK_SNIPPET_CHARS
os.environ["RAG_TEMPERATURE_RERANK"] = TEMPERATURE_RERANK
os.environ["RAG_TEMPERATURE_ANSWER"] = TEMPERATURE_ANSWER
os.environ["RAG_MAX_TOKENS_ANSWER"] = MAX_TOKENS_ANSWER
os.environ["RAG_MAX_RETRIES"] = MAX_RETRIES
os.environ["RAG_RETRY_SLEEP_SECS"] = RETRY_SLEEP_SECS
# os.environ["DATABRICKS_TOKEN"] = token # Acceso para el service endpoint al search index 
# os.environ["DATABRICKS_HOST"] = host # Acceso para el service endpoint al search index 

# env usadas en el serving endpoint
env_vars = {
    "RAG_LLM_ENDPOINT": os.environ["RAG_LLM_ENDPOINT"],
    "RAG_EMBED_ENDPOINT": os.environ["RAG_EMBED_ENDPOINT"],
    "RAG_VS_ENDPOINT": os.environ["RAG_VS_ENDPOINT"],
    "RAG_VS_INDEX": os.environ["RAG_VS_INDEX"],
    "RAG_VS_INDEX_FULL_NAME": os.environ["RAG_VS_INDEX_FULL_NAME"],
    "RAG_TOP_K_CANDIDATES": os.environ["RAG_TOP_K_CANDIDATES"],
    "RAG_TOP_K_FINAL": os.environ["RAG_TOP_K_FINAL"],
    "RAG_LEX_FALLBACK_LIMIT": os.environ["RAG_LEX_FALLBACK_LIMIT"],
    "RAG_MAX_CONTEXT_CHARS": os.environ["RAG_MAX_CONTEXT_CHARS"],
    "RAG_RERANK_SNIPPET_CHARS": os.environ["RAG_RERANK_SNIPPET_CHARS"],
    "RAG_TEMPERATURE_RERANK": os.environ["RAG_TEMPERATURE_RERANK"],
    "RAG_TEMPERATURE_ANSWER": os.environ["RAG_TEMPERATURE_ANSWER"],
    "RAG_MAX_TOKENS_ANSWER": os.environ["RAG_MAX_TOKENS_ANSWER"],
    "RAG_MAX_RETRIES": os.environ["RAG_MAX_RETRIES"],
    "RAG_RETRY_SLEEP_SECS": os.environ["RAG_RETRY_SLEEP_SECS"],
    # "DATABRICKS_TOKEN": os.environ["DATABRICKS_TOKEN"],
    # "DATABRICKS_HOST": os.environ["DATABRICKS_HOST"],
}

In [0]:
# ==============================
# 3) Localizar el paquete bind_rag_agent
# ==============================

if BUNDLE_FILE_PATH:
    project_root = Path(BUNDLE_FILE_PATH).parent
else:
    project_root = Path(f"/Workspace/Users/{current_user}/bind_agent")

bind_rag_pkg = project_root / "src" / "bind_rag_agent"
assert bind_rag_pkg.exists(), f"No se encontro bind_rag_agent en {bind_rag_pkg}"

# Verificar que tiene __init__.py
assert (bind_rag_pkg / "__init__.py").exists(), f"Falta __init__.py en {bind_rag_pkg}"

print(f"Package encontrado: {bind_rag_pkg}")
print(f"Archivos: {[f.name for f in bind_rag_pkg.iterdir() if f.suffix == '.py']}")


In [0]:
# ==============================
# 4) Log + Register (UC) - ChatModel (streaming nativo)
# ==============================
import sys
import time
import re
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex
from mlflow.pyfunc import ChatModel
from mlflow.types.llm import (
    ChatCompletionResponse,
    ChatCompletionChunk,
    ChatMessage,
    ChatChoice,
    ChatParams,
    ChatChoiceDelta,
    ChatChunkChoice,
)
from typing import List, Generator, Optional

resources = [
    DatabricksServingEndpoint(endpoint_name=os.environ["RAG_LLM_ENDPOINT"]),
    DatabricksServingEndpoint(endpoint_name=os.environ["RAG_EMBED_ENDPOINT"]),
    DatabricksVectorSearchIndex(index_name=os.environ["RAG_VS_INDEX_FULL_NAME"]),
]

w = WorkspaceClient()
w.workspace.mkdirs(f"/Users/{current_user}/bind_agent")

mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.set_registry_uri("databricks-uc")

# Agregar el parent de bind_rag_agent al sys.path para que
# MLflow pueda validar el modelo durante log_model
pkg_parent = str(bind_rag_pkg.parent)
if pkg_parent not in sys.path:
    sys.path.insert(0, pkg_parent)


class RagChatModel(ChatModel):
    """
    ChatModel habilita streaming nativo en serving endpoints.
    """

    def load_context(self, context):
        from bind_rag_agent.rag_agent import RagAgent
        self._agent = RagAgent()

    # def _run_rag(self, query: str) -> str:
    #     import pandas as _pd
    #     if not hasattr(self, '_agent'):
    #         self.load_context(None)
    #     try:
    #         df = _pd.DataFrame([{"query": query}])
    #         out = self._agent.predict(None, df)
    #         if isinstance(out, list) and out:
    #             answer = out[0].get("answer", "")
    #             error = out[0].get("error", "")
    #             if error:
    #                 return f"Error: {error}"
    #             return answer or "No se encontro informacion relevante."
    #         return "Sin respuesta del RAG."
    #     except Exception as e:
    #         return f"Error procesando la consulta: {e}"

    def _run_rag(self, query: str) -> str:
        import pandas as _pd
        import json as _json
        if not hasattr(self, '_agent'):
            self.load_context(None)
        try:
            df = _pd.DataFrame([{"query": query}])
            out = self._agent.predict(None, df)
            if isinstance(out, list) and out:
                answer = out[0].get("answer", "")
                error = out[0].get("error", "")
                if error:
                    return f"Error: {error}"
                if not answer:
                    return "No se encontro informacion relevante."

                # Agregar fuentes al final de la respuesta
                sources_raw = out[0].get("sources_json", "[]")
                try:
                    sources = _json.loads(sources_raw) if isinstance(sources_raw, str) else sources_raw
                except Exception:
                    sources = []

                if sources:
                    answer += "\n\n📎 **Fuentes:**\n"
                    for s in sources:
                        parts = []
                        if s.get("path"):
                            # Solo el nombre del archivo
                            fname = s["path"].rsplit("/", 1)[-1]
                            parts.append(fname)
                        if s.get("page_num") is not None:
                            parts.append(f"p.{s['page_num']}")
                        if s.get("topic"):
                            parts.append(s["topic"])
                        answer += f"- [{s.get('sid', '?')}] {' | '.join(parts)}\n"

                return answer
            return "Sin respuesta del RAG."
        except Exception as e:
            return f"Error procesando la consulta: {e}"

    def predict(
        self, context, messages: List[ChatMessage], params: ChatParams
    ) -> ChatCompletionResponse:
        query = messages[-1].content if messages else ""
        content = self._run_rag(query) if query else "No se recibio ninguna pregunta."
        return ChatCompletionResponse(
            choices=[
                ChatChoice(
                    message=ChatMessage(role="assistant", content=content)
                )
            ]
        )

    def _create_chunk(self, content: str, finish_reason: Optional[str] = None) -> ChatCompletionChunk:
        return ChatCompletionChunk(
            choices=[
                ChatChunkChoice(
                    delta=ChatChoiceDelta(role="assistant", content=content),
                    finish_reason=finish_reason,
                )
            ]
        )

    def predict_stream(
        self, context, messages: List[ChatMessage], params: ChatParams
    ) -> Generator[ChatCompletionChunk, None, None]:
        query = messages[-1].content if messages else ""
        content = self._run_rag(query) if query else "No se recibio ninguna pregunta."

        chunk_size = 100
        for i in range(0, len(content), chunk_size):
            yield self._create_chunk(content[i:i+chunk_size])

        yield self._create_chunk("", finish_reason="stop")


with mlflow.start_run(run_name="rag_agent_deploy") as run:
    logged = mlflow.pyfunc.log_model(
        artifact_path="rag_agent",
        python_model=RagChatModel(),
        code_paths=[str(bind_rag_pkg)],
        extra_pip_requirements=["databricks-vectorsearch"],
        resources=resources,
    )
    model_uri = logged.model_uri
    print("Model logged:", model_uri)

    uc_model_info = mlflow.register_model(model_uri=model_uri, name=UC_MODEL_NAME)
    print("Registered in UC:", uc_model_info.name, "version:", uc_model_info.version)

# Esperar READY
client = MlflowClient()
t0 = time.time()
while True:
    mv = client.get_model_version(name=UC_MODEL_NAME, version=uc_model_info.version)
    if mv.status == "READY":
        break
    if time.time() - t0 > 900:
        raise TimeoutError(f"Timeout esperando READY para {UC_MODEL_NAME} v{uc_model_info.version}")
    print("Esperando model READY... status=", mv.status)
    time.sleep(5)

print(f"Model version READY: {UC_MODEL_NAME} v{uc_model_info.version}")

# Verificar task
model_info = mlflow.models.get_model_info(model_uri)
print(f"Task metadata: {model_info.metadata}")


In [0]:
# ==============================
# 5) Crear/Actualizar Model Serving Endpoint
# ==============================
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
import datetime as dt
from databricks.sdk.errors import NotFound

def ensure_serving_endpoint(endpoint_name, model_name, model_version, workload_size="Small"):
    served = ServedEntityInput(
        name=f"{endpoint_name}-entity",
        entity_name=model_name,
        entity_version=str(model_version),
        workload_size=workload_size,
        scale_to_zero_enabled=True,
        environment_vars=env_vars,
    )

    cfg = EndpointCoreConfigInput(name=endpoint_name, served_entities=[served])

    try:
        w.serving_endpoints.get(endpoint_name)
        print(f"[serving] Endpoint existe: {endpoint_name}")
        w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[served],
            timeout=dt.timedelta(minutes=30),
        )
    except NotFound:
        print(f"[serving] Creando endpoint: {endpoint_name}")
        w.serving_endpoints.create_and_wait(
            name=endpoint_name,
            config=cfg,
            timeout=dt.timedelta(minutes=30),
        )

    print("✅ Serving endpoint listo:", endpoint_name)

ensure_serving_endpoint(
    endpoint_name=MODEL_SERVING_ENDPOINT,
    model_name=UC_MODEL_NAME,
    model_version=uc_model_info.version,
    workload_size="Small",
)

In [0]:
# ==============================
# 6) Smoke test (invocar endpoint) - formato chat
# ==============================
from mlflow.deployments import get_deploy_client
dc = get_deploy_client("databricks")

payload = {
    "messages": [
        {"role": "user", "content": "Cuales son las Previsiones de octubre 2025?"}
    ]
}

resp = dc.predict(endpoint=MODEL_SERVING_ENDPOINT, inputs=payload)
print(resp)


In [0]:
# ==============================
# 7) Nada que limpiar (sin directorio temporal)
# ==============================
print("Deploy completo.")


In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
ep = w.serving_endpoints.get(MODEL_SERVING_ENDPOINT)
print(f"Task: {ep.task}")